# RTV Solver — Quick Tutorial

This notebook walks through the online dispatching loop of the RTV (Request–Trip–Vehicle)
solver on a synthetic Wilson, NC sample.

**Before you start:**
- Run from the repository root (the package is used standalone — no install needed beyond
  `pip install -r requirements.txt`).
- An **OSRM** server must be running and reachable at the URL used below
  (see the README's *Set up the OSRM server* section).

The flow: load a day of requests → dispatch them in short time batches → advance
(simulate) the fleet between batches → inspect feasibility, fallbacks, and service metrics.

## 1. Load a sample payload

A **payload** is a dict with three keys: `depot`, `requests`, and `driver_runs`
(the vehicles). All times are **integer seconds since midnight**.

In [1]:
import pickle
import warnings
warnings.filterwarnings("ignore")  # silence unrelated pandas/bottleneck notices

with open('inputs/wilson/sample1.pkl', 'rb') as f:
    payload = pickle.load(f)

print('requests   :', len(payload['requests']))
print('vehicles   :', len(payload['driver_runs']))
print('depot      :', payload['depot']['pt'])
payload['requests'][0]

requests   : 100
vehicles   : 4
depot      : {'lat': 35.721332, 'lon': -77.915633}


{'booking_id': '1',
 'am': 1,
 'wc': 0,
 'pickup_pt': {'lat': 35.695793, 'lon': -77.888079},
 'dropoff_pt': {'lat': 35.746108, 'lon': -77.934598},
 'pickup_time_window_start': 54068,
 'pickup_time_window_end': 55868,
 'dropoff_time_window_start': 54876.4,
 'dropoff_time_window_end': 56676.4}

## 2. Initialize the solver

`OnlineRTVSolver` solves one batch of requests at a time. Point it at your OSRM server.

In [2]:
from rtv_solver import OnlineRTVSolver

online_rtv_solver = OnlineRTVSolver("http://127.0.0.1:50000/")

## 3. Dispatch the first batch

Real-time dispatching processes requests in short windows. Here we take every request
whose pickup window opens before **06:20**, then assign them.

Two assignment methods are available:
- `solve_pdptw_heuristic` — fast insertion heuristic (used here for the first batch)
- `solve_pdptw_rtv` — full RTV assignment via the Gurobi ILP (used for the next batch)

Both return `(driver_runs, unserved_requests)`.

In [3]:
current_time = 6*3600           # 06:00:00
step = 20*60                    # 20-minute batches

selected_requests = [r for r in payload['requests']
                     if r['pickup_time_window_start'] < current_time + step]
len(selected_requests)

1

In [4]:
new_payload = {
    "depot": payload["depot"],
    "requests": selected_requests,
    "driver_runs": payload["driver_runs"],
}

# Fast insertion heuristic
driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_heuristic(new_payload)
print('unserved:', unserved_requests)

unserved: []


### Advance the fleet

`simulate_manifest` rolls the vehicles forward to a new clock time. Stops whose scheduled
time has passed become history (they are frozen and never re-planned).

In [5]:
current_time += step   # 06:20:00
simulated_driver_runs = online_rtv_solver.simulate_manifest(
    current_time, driver_runs, intermediate_location=False)

## 4. Dispatch the next batch with full RTV

Now take the requests that open between **06:20** and **06:40** and assign them on top of
the fleet's current state using the RTV ILP.

In [6]:
selected_requests = [r for r in payload['requests']
                     if current_time <= r['pickup_time_window_start'] < current_time + step]

new_payload = {
    "depot": payload["depot"],
    "requests": selected_requests,
    "driver_runs": simulated_driver_runs,
}

# Full RTV assignment
driver_runs, unserved_requests = online_rtv_solver.solve_pdptw_rtv(new_payload)
print('batch size:', len(selected_requests), '| unserved:', unserved_requests)

current_time += step   # 06:40:00
simulated_driver_runs = online_rtv_solver.simulate_manifest(
    current_time, driver_runs, intermediate_location=False)

batch size: 1 | unserved: []


## 5. Check feasibility of candidate time windows

Before committing a request you can ask which of several candidate windows are feasible,
each scored by its VMT/PMT ratio (vehicle-miles vs. passenger-miles — lower is a tighter,
more efficient fit). This does **not** commit the request.

In [7]:
next_batch = [r for r in payload['requests']
              if current_time <= r['pickup_time_window_start'] < current_time + step]
req = next_batch[0]

probe = {
    "requests": [{
        'booking_id': req['booking_id'],
        'pickup_pt': req['pickup_pt'],
        'dropoff_pt': req['dropoff_pt'],
        'am': req['am'], 'wc': req['wc'],
        'time_windows': [
            {'pickup_time_window_start': req['pickup_time_window_start'],
             'pickup_time_window_end':   req['pickup_time_window_start'] + 60,
             'dropoff_time_window_start': req['dropoff_time_window_start'],
             'dropoff_time_window_end':   req['dropoff_time_window_start'] + 180},
            {'pickup_time_window_start': req['pickup_time_window_start'] + 900,
             'pickup_time_window_end':   req['pickup_time_window_end']   + 900,
             'dropoff_time_window_start': req['dropoff_time_window_start'] + 900,
             'dropoff_time_window_end':   req['dropoff_time_window_end']   + 900},
            {'pickup_time_window_start': req['pickup_time_window_start'] + 1800,
             'pickup_time_window_end':   req['pickup_time_window_end']   + 1800,
             'dropoff_time_window_start': req['dropoff_time_window_start'] + 1800,
             'dropoff_time_window_end':   req['dropoff_time_window_end']   + 1800},
        ],
    }],
    "depot": payload["depot"],
    "driver_runs": simulated_driver_runs,
}

feasible_windows = online_rtv_solver.check_feasibility(probe)
feasible_windows   # -> [(time_window, vmt/pmt ratio), ...]

[({'pickup_time_window_start': 24449,
   'pickup_time_window_end': 24509,
   'dropoff_time_window_start': 24954.4,
   'dropoff_time_window_end': 25134.4},
  1.293177627535341),
 ({'pickup_time_window_start': 25349,
   'pickup_time_window_end': 27149,
   'dropoff_time_window_start': 25854.4,
   'dropoff_time_window_end': 27654.4},
  1.293177627535341),
 ({'pickup_time_window_start': 26249,
   'pickup_time_window_end': 28049,
   'dropoff_time_window_start': 26754.4,
   'dropoff_time_window_end': 28554.4},
  1.293177627535341)]

## 6. Unserved requests and *serve ASAP*

If a request's window is too tight to fit, RTV returns it in `unserved_requests`. Below we
build a deliberately tight window, see it come back unserved, then fall back to
`serve_asap`, which inserts it at its earliest feasible pickup time (ignoring cost).

In [8]:
tight = {
    "depot": payload["depot"],
    "requests": [{
        'booking_id': req['booking_id'],
        'pickup_pt': req['pickup_pt'],
        'dropoff_pt': req['dropoff_pt'],
        'am': req['am'], 'wc': req['wc'],
        'pickup_time_window_start': req['pickup_time_window_start'],
        'pickup_time_window_end':   req['pickup_time_window_start'] + 180,
        # Impossible dropoff window: must arrive within 60s of the pickup opening.
        'dropoff_time_window_start': req['pickup_time_window_start'],
        'dropoff_time_window_end':   req['pickup_time_window_start'] + 60,
    }],
    "driver_runs": simulated_driver_runs,
}

_, unserved_requests = online_rtv_solver.solve_pdptw_rtv(tight)
print('unserved by RTV :', unserved_requests)

asap_driver_runs, still_unserved = online_rtv_solver.serve_asap(tight)
print('unserved by ASAP:', still_unserved)

unserved by RTV : ['11']


unserved by ASAP: []


## 7. Re-optimize committed runs

`resolve_pdptw_rtv` re-solves the committed manifests with no new requests. It returns the
improved runs only if every request stays served (otherwise the input is returned
unchanged).

In [9]:
reoptimized = online_rtv_solver.resolve_pdptw_rtv(
    {"depot": payload["depot"], "driver_runs": asap_driver_runs})
print('vehicles:', len(reoptimized))

vehicles: 4


## 8. Service metrics

`get_stats` validates a set of runs and returns `(feasible, stats)`.

In [10]:
feasible, stats = online_rtv_solver.get_stats(payload["depot"], simulated_driver_runs)
print('feasible          :', feasible)
print('serviced          :', stats['serviced'])
print('VMT (s)           : %.0f' % stats['vmt'])
print('PMT (s)           : %.0f' % stats['pmt'])
print('VMT/PMT           : %.2f' % stats['vmt/pmt'])
print('avg wait (s)      : %.0f' % stats['average_wait_time'])
print('avg detour (s)    : %.0f' % stats['average_detour'])

feasible          : True
serviced          : 2
VMT (s)           : 1358
PMT (s)           : 802
VMT/PMT           : 1.69
avg wait (s)      : 101
avg detour (s)    : 180


## Doing it all at once

To run a whole day instead of hand-driving each batch, use `OfflineRTVSolver`, or the
command-line scripts:

```
python scripts/generate_sample_data.py   # make synthetic inputs
python scripts/run_solver.py             # solve them -> outputs/
python scripts/analyze_output.py         # service-rate / VMT / PMT / occupancy / ...
```

See the README for the full API and the payload schema.